In [ ]:
import numpy as np
import mpmath
import matplotlib.pyplot as plt
import pygridsynth as gridsynth

from bloqade import squin
from bloqade.cirq_utils import emit_circuit
import cirq

def string_to_unitary(gate_string):
    @squin.kernel
    def circ():
        q = squin.qalloc(1)
        for char in gate_string:
            if char == "H":
                squin.h(q[0])
            elif char == "S":
                squin.s(q[0])
            elif char == "T":
                squin.t(q[0])
            elif char == "X":
                squin.x(q[0])
            elif char == "Y":
                squin.y(q[0])
            elif char == "Z":
                squin.z(q[0])
            # We ignore "W" since it is just a global phase and our distance metric is invariant to it.
        return q
        
    c = emit_circuit(circ, ignore_returns=True)
    # Handle empty circuits safely
    if not c.all_qubits():
        return np.eye(2, dtype=complex)
    return cirq.unitary(c)

def rz(theta):
    return np.array([[np.exp(-1j * theta / 2), 0],
                     [0, np.exp(1j * theta / 2)]], dtype=complex)

def distance(U, V):
    tr = np.trace(U.conj().T @ V)
    val = 1 - np.abs(tr)/2
    if val < 0: val = 0
    return np.sqrt(val)

In [ ]:
mpmath.mp.dps = 128
epsilon = mpmath.mpf("1e-4")

exact_gates = {0: "Z", 1: "S", 2: "T"}

results = []
for n in range(6):
    theta_val = np.pi / (2**n)
    theta_mp = mpmath.mpf(mpmath.pi) / (2**n)
    
    if n in exact_gates:
        gates = exact_gates[n]
    else:
        # Synthesize with pygridsynth
        gates = gridsynth.gridsynth_gates(theta=theta_mp, epsilon=epsilon)
    
    # Evaluate distance
    U_approx = string_to_unitary(gates)
    U_target = rz(theta_val)
    dist = distance(U_target, U_approx)
    
    t_count = gates.count("T")
    results.append({
        "n": n,
        "gates": gates,
        "length": len(gates),
        "t_count": t_count,
        "distance": dist
    })
    print(f"n={n}: target Rz(pi/{2**n})")
    print(f"  Distance: {dist:.2e}")
    print(f"  T-count:  {t_count}")
    print(f"  Length:   {len(gates)}")
    print(f"  Sequence: {gates}\n")


In [ ]:
# Visualizing T-count vs Distance (epsilon scaling) for non-Clifford rotations
epsilons = [mpmath.mpf(f"1e-{i}") for i in range(1, 6)]

# We skip n=0, 1, 2 as they can be exactly implemented with 0 or 1 T-gates
n_values = [3, 4, 5]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes = axes.flatten()

for i, n_val in enumerate(n_values):
    distances = []
    t_counts = []

    theta_mp = mpmath.mpf(mpmath.pi) / (2**n_val)
    theta_val = np.pi / (2**n_val)
    U_target = rz(theta_val)

    for eps in epsilons:
        gates = gridsynth.gridsynth_gates(theta=theta_mp, epsilon=eps)
        U_approx = string_to_unitary(gates)
        dist = distance(U_target, U_approx)
        
        distances.append(dist)
        t_counts.append(gates.count("T"))

    ax = axes[i]
    ax.plot(distances, t_counts, marker="o", linestyle="--")
    
    if all(d > 0 for d in distances):
        ax.set_xscale("log")
    
    ax.set_xlabel("Actual Distance $d(U, V)$")
    ax.set_ylabel("T-count")
    ax.set_title(f"T-count vs Distance for $R_z(\\pi/2^{{{n_val}}})$ (n={n_val})")
    ax.grid(True, which="both", ls="--")

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 4 ── Teleportazione con gate V Clifford+T ──────────────────────────
#
# Protocollo:
#   q0: stato da teleportare   |ψ⟩ = α|0⟩ + β|1⟩
#   q1: risorsa               Rz(θ)|+⟩
#
#   1) CNOT(q1→q0)
#   2) Misura q0  →  m ∈ {0,1}
#   3) Se m==1: applica V = Rz(θ)·X·Rz(-θ)  su q1
#
# Qui V viene costruito sia con cirq.rz (esatto) sia con la sequenza
# Clifford+T sintetizzata da pygridsynth, e i due risultati sono confrontati.

import numpy as np
import mpmath
import cirq
from rich.console import Console
from rich.panel import Panel
from rich.table import Table

console = Console()

# ── Stato iniziale q0 ──────────────────────────────────────────────────────
alpha = complex(1, 1)
beta  = complex(1, -1)
norm  = np.sqrt(abs(alpha)**2 + abs(beta)**2)
alpha /= norm
beta  /= norm

console.print(f"[bold]Stato iniziale q0:[/bold] α|0⟩ + β|1⟩")
console.print(f"  α = {np.round(alpha, 4)}")
console.print(f"  β = {np.round(beta,  4)}\n")

q0 = cirq.LineQubit(0)
q1 = cirq.LineQubit(1)
simulator = cirq.Simulator()

# ── Helper: costruisce ops di V con metodo esatto ─────────────────────────
def gate_V_exact(qubit, theta_float):
    """V = Rz(θ)·X·Rz(-θ)  usando cirq.rz esatto."""
    return [
        cirq.rz(theta_float)(qubit),
        cirq.X(qubit),
        cirq.rz(-theta_float)(qubit),
    ]

# ── Helper: costruisce ops di V con sequenza Clifford+T ───────────────────
def gate_V_synth(qubit, gate_string_pos, gate_string_neg):
    """
    V ≈ Rz_synth(θ) · X · Rz_synth(-θ)
    gate_string_pos: sequenza per Rz(+θ)
    gate_string_neg: sequenza per Rz(-θ)  (= Rz(+θ)† = reverse + dagger)
    """
    ops = []
    for char in gate_string_pos:
        if char == "H": ops.append(cirq.H(qubit))
        elif char == "S": ops.append(cirq.S(qubit))
        elif char == "T": ops.append(cirq.T(qubit))
        elif char == "X": ops.append(cirq.X(qubit))
        elif char == "Y": ops.append(cirq.Y(qubit))
        elif char == "Z": ops.append(cirq.Z(qubit))
    ops.append(cirq.X(qubit))   # la X centrale di V
    for char in gate_string_neg:
        if char == "H": ops.append(cirq.H(qubit))
        elif char == "S": ops.append(cirq.S(qubit))
        elif char == "T": ops.append(cirq.T(qubit))
        elif char == "X": ops.append(cirq.X(qubit))
        elif char == "Y": ops.append(cirq.Y(qubit))
        elif char == "Z": ops.append(cirq.Z(qubit))
    return ops


# ── Helper: ottiene la sequenza per Rz(-θ) invertendo quella per Rz(+θ) ──
def invert_gate_string(gate_string):
    """
    Rz(-θ) ≈ [Rz(+θ)]†.
    Per un prodotto di gate Clifford+T (ciascuno self-inverse o con inverso noto):
      (A·B·C)† = C†·B†·A†
    Inversi: H→H, X→X, Y→Y, Z→Z, S→S† (=S³=SZ), T→T† (=T⁷=TSTSSTS)
    Approssimazione pratica: si usa gridsynth direttamente su -θ.
    Qui usiamo il dizionario synth_gates già calcolato solo per il ramo θ>0;
    per -θ richiamiamo gridsynth.
    """
    # Nota: gridsynth sintetizza angoli arbitrari; lo usiamo nella Cell 4 per -θ
    raise NotImplementedError("Usa synth_gates_neg dal ciclo principale")


# ── Loop principale ────────────────────────────────────────────────────────
mpmath.mp.dps = 128
epsilon_tele = mpmath.mpf("1e-4")
exact_gates  = {0: "Z", 1: "S", 2: "T"}

# Pre-calcola sequenze per +θ e -θ
# Rz(-θ) = Rz(2π - θ) a meno di fase globale; oppure si sintetizza direttamente
def get_synth(theta_mp, eps):
    return gridsynth.gridsynth_gates(theta=theta_mp, epsilon=eps)

for n in range(6):
    theta_float = np.pi / (2**n)
    theta_mp    = mpmath.mpf(mpmath.pi) / (2**n)
    denom       = 2**n

    # Sequenza sintetizzata per +θ e -θ
    if n in exact_gates:
        gates_pos = exact_gates[n]
    else:
        gates_pos = get_synth(theta_mp, epsilon_tele)

    # Per -θ: gridsynth su -theta_mp  (o 2π - theta_mp, equivalente a meno di fase)
    if n == 0:
        gates_neg = "Z"   # Rz(-π) = Z a fase globale
    elif n == 1:
        gates_neg = "SSS" # S† = S³
    elif n == 2:
        gates_neg = "TTTTTTT"  # T† = T⁷
    else:
        gates_neg = get_synth(-theta_mp % (2 * mpmath.pi), epsilon_tele)

    # Distanza di V_synth rispetto a V_exact
    U_rz_pos   = rz(theta_float)
    U_rz_neg   = rz(-theta_float)
    U_syn_pos  = string_to_unitary(gates_pos)
    U_syn_neg  = string_to_unitary(gates_neg)

    X_mat = np.array([[0,1],[1,0]], dtype=complex)
    V_exact = U_rz_pos @ X_mat @ U_rz_neg
    V_synth = U_syn_pos @ X_mat @ U_syn_neg
    dist_V  = distance(V_exact, V_synth)

    # ── Stato a 2 qubit: |ψ⟩_q0 ⊗ Rz(θ)|+⟩_q1 ─────────────────────────
    plus   = np.array([1, 1], dtype=complex) / np.sqrt(2)
    q1_init = U_rz_pos @ plus          # Rz(θ)|+⟩
    gamma, delta = q1_init[0], q1_init[1]

    initial_state = np.array([
        alpha * gamma,  # |00⟩
        alpha * delta,  # |01⟩
        beta  * gamma,  # |10⟩
        beta  * delta,  # |11⟩
    ], dtype=complex)

    # ── Circuito di esempio (solo per visualizzazione, usa esatto) ────────
    circuit_display = cirq.Circuit([
        cirq.CNOT(q1, q0),
        cirq.measure(q0, key='m'),
        *[op.with_classical_controls('m') for op in gate_V_exact(q1, theta_float)],
    ])

    console.print(Panel(
        str(circuit_display),
        title=f"[bold cyan]θ = π/{denom}  (n={n})[/bold cyan]",
        subtitle=f"[dim]gates(+θ): {gates_pos[:30]}{'...' if len(gates_pos)>30 else ''}   "
                 f"T-count: {gates_pos.count('T')}   d(V_exact, V_synth)={dist_V:.2e}[/dim]",
        border_style="cyan"
    ))

    # ── Passo 1: CNOT + misura q0 ─────────────────────────────────────────
    circuit_pre = cirq.Circuit([
        cirq.CNOT(q1, q0),
        cirq.measure(q0, key='m'),
    ])
    result_pre = simulator.simulate(
        circuit_pre,
        initial_state=initial_state,
        qubit_order=[q0, q1],
    )
    m = result_pre.measurements['m'][0]
    state_after_measure = result_pre.final_state_vector

    # ── Passo 2a: V esatto ────────────────────────────────────────────────
    if m == 1:
        c_exact = cirq.Circuit(*gate_V_exact(q1, theta_float))
    else:
        c_exact = cirq.Circuit(cirq.I(q1))

    res_exact = simulator.simulate(c_exact, initial_state=state_after_measure,
                                   qubit_order=[q0, q1])
    sv_e = res_exact.final_state_vector.reshape(2, 2)
    rho_e = np.einsum('ij,ik->jk', sv_e, sv_e.conj())
    ev_e, evec_e = np.linalg.eigh(rho_e)
    q1_exact = evec_e[:, np.argmax(ev_e)]

    # ── Passo 2b: V sintetizzato (Clifford+T) ────────────────────────────
    if m == 1:
        ops_synth = gate_V_synth(q1, gates_pos, gates_neg)
        c_synth   = cirq.Circuit(*ops_synth)
    else:
        c_synth = cirq.Circuit(cirq.I(q1))

    res_synth = simulator.simulate(c_synth, initial_state=state_after_measure,
                                   qubit_order=[q0, q1])
    sv_s = res_synth.final_state_vector.reshape(2, 2)
    rho_s = np.einsum('ij,ik->jk', sv_s, sv_s.conj())
    ev_s, evec_s = np.linalg.eigh(rho_s)
    q1_synth = evec_s[:, np.argmax(ev_s)]

    # ── Distanza tra i due stati finali di q1 ────────────────────────────
    # Usando la fedeltà: F = |⟨ψ_exact|ψ_synth⟩|²
    fidelity = abs(np.vdot(q1_exact, q1_synth))**2
    infidelity = 1 - fidelity

    # ── Tabella riassuntiva ───────────────────────────────────────────────
    table = Table(show_header=True, header_style="bold magenta")
    table.add_column("")
    table.add_column("q1 finale  (coeff |0⟩)")
    table.add_column("q1 finale  (coeff |1⟩)")

    table.add_row(
        "[green]Esatto[/green]",
        str(np.round(q1_exact[0], 5)),
        str(np.round(q1_exact[1], 5)),
    )
    table.add_row(
        "[yellow]Clifford+T[/yellow]",
        str(np.round(q1_synth[0], 5)),
        str(np.round(q1_synth[1], 5)),
    )

    console.print(table)
    console.print(f"  Misura q0 = {m}")
    console.print(f"  d(V_exact, V_synth)  = {dist_V:.2e}")
    console.print(f"  Infedeltà stato q1   = {infidelity:.2e}")
    console.print(f"  T-count (+θ)         = {gates_pos.count('T')}")
    console.print(f"  T-count (-θ)         = {gates_neg.count('T')}\n")

In [ ]:
def process_fidelity(U, V):
    """F(U,V) = |Tr(U†V)|² / 4  per gate su 1 qubit."""
    return (abs(np.trace(U.conj().T @ V)) ** 2) / 4

def hs_norm(U, V):
    """Norma di Hilbert-Schmidt: sensibile a fase globale."""
    diff = U - V
    return np.sqrt(abs(np.trace(diff.conj().T @ diff)))